# Módulo 4 — Ensamblaje de la cápside
### BruSafe / CAPSID · iGEM 2026

Cascada explícita de nucleación-elongación sobre los **90 tamaños intermedios** de la cápside T=3.
A diferencia de un modelo de tres estados, el modelo sigue cada oligómero de 1 a 90 dímeros.

Este notebook produce tres resultados que un modelo de tres estados no puede producir:

1. el periodo de latencia de nucleación
2. la **trampa cinética**, dependiente de la concentración
3. un **óptimo** de concentración inicial de dímero, que constituye la predicción falsable de M4

Se ejecuta con `Runtime → Run all`. El barrido tarda aproximadamente 1 min.

## 1 · Preparación

In [ ]:
# ---------------------------------------------------------------
# Solo se usan numpy, scipy y matplotlib, que ya vienen instalados en Colab.
# ---------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

RT    = 0.593    # kcal/mol a ~298 K
N_CAP = 90       # dimeros de coat protein por capside T=3 (Ni 1995)

## 2 · Parámetros

In [ ]:
# Parametros de ensamblaje: (valor central, (min, max), fuente)
PARAMS_M4 = {
    "k_a_uM_min":     (3.0,  (0.4, 4.0),  "Ceres 2002; Endres 2002"),
    "g_assoc_kcal":   (6.0,  (4.0, 8.0),  "Zlotnick 1994 (penalizacion entropica)"),
    "g_contact_kcal": (-4.0, (-5., -3.),  "Ceres 2002 (energia por contacto)"),
    "n_c_nucleus":    (4,    (3, 6),      "Endres 2002 (tamano del nucleo)"),
}

def central(params=PARAMS_M4):
    return {k: v[0] for k, v in params.items()}

q = central()
for k, (v, rng, src) in PARAMS_M4.items():
    print(f"{k:<16} {v:<7} rango {str(rng):<13} {src}")

## 3 · El modelo

Las constantes **inversas** no se ajustan libremente: se obtienen por balance detallado a partir de
las energías de contacto. Por esta razón el modelo no puede reproducir el óptimo ajustando los
parámetros de forma independiente.

In [ ]:
def contacts_for_step(j_target, n_c):
    """Contactos subunidad-subunidad que se forman al añadir una subunidad
    para llegar al j_target-mero.

    Toda la física del modelo está en esta función:

      pre-núcleo (j <= n_c):  1 contacto  -> con la penalización entrópica el
          paso es netamente desfavorable. Esa es la barrera de nucleación y la
          causa del periodo de latencia.
      crecimiento:            2 contactos -> favorable, elongación rápida.
      cierre (últimos ~8):    3 contactos -> muy favorable; el cierre de los
          vértices empuja con fuerza hacia la cápside completa
          (Endres & Zlotnick 2002).
    """
    if j_target <= n_c:       return 1
    if j_target >= N_CAP - 8: return 3
    return 2


def build_rates(k_a_uM_min, g_assoc_kcal, g_contact_kcal, n_c_nucleus):
    """Tasas directas e inversas.

    La tasa directa k_a es la misma para todos los pasos. Las tasas inversas
    no se ajustan libremente: se obtienen por balance detallado a partir de
    las energías de contacto, lo que impide ajustarlas de forma independiente.

      dG_add(j) = g_assoc + contactos(j) * g_contact
      Kd(j)     = exp(dG_add / RT)
      b_{j+1}   = k_a * Kd(j)
    """
    f = np.full(N_CAP, k_a_uM_min)
    b = np.zeros(N_CAP + 1)
    for j in range(1, N_CAP):
        c = contacts_for_step(j + 1, n_c_nucleus)
        dG_add = g_assoc_kcal + c * g_contact_kcal
        b[j + 1] = k_a_uM_min * np.exp(dG_add / RT)
    return f, b


def cascade_ode(t, x, f, b):
    """x[j-1] = concentracion (uM) del j-mero, j = 1..90.
    x[0] es el DIMERO libre de coat protein: la subunidad de ensamblaje.
    Reaccion j:  X_j + X_1 <-> X_{j+1}   (j=1 usa x_1^2)
    """
    x = np.clip(x, 0.0, None)
    x1 = x[0]
    J = np.zeros(N_CAP - 1)
    J[0] = f[0] * x1 * x1 - b[2] * x[1]
    for j in range(2, N_CAP):
        J[j - 1] = f[j - 1] * x[j - 1] * x1 - b[j + 1] * x[j]
    dx = np.zeros(N_CAP)
    dx[0] = -2 * J[0] - np.sum(J[1:])
    dx[1] = J[0] - J[1]
    for k in range(3, N_CAP):
        dx[k - 1] = J[k - 2] - J[k - 1]
    dx[N_CAP - 1] = J[N_CAP - 2]
    return dx


def simulate_cascade(D0_uM, k_a_uM_min, g_assoc_kcal, g_contact_kcal,
                     n_c_nucleus, t_max=240.0, nt=600):
    """Devuelve t, capside(t), libre(t), intermedios(t), estado completo.
    Las fracciones son de masa del pool total de subunidades y suman 1."""
    f, b = build_rates(k_a_uM_min, g_assoc_kcal, g_contact_kcal, n_c_nucleus)
    x0 = np.zeros(N_CAP); x0[0] = D0_uM
    t_eval = np.linspace(0, t_max, nt)
    sol = solve_ivp(cascade_ode, (0, t_max), x0, t_eval=t_eval, args=(f, b),
                    method="LSODA", rtol=1e-7, atol=1e-10)
    x = sol.y
    j = np.arange(1, N_CAP + 1)[:, None]
    capsid = N_CAP * x[-1] / D0_uM
    free   = x[0] / D0_uM
    inter  = (np.sum(j * x, axis=0) - x[0] - N_CAP * x[-1]) / D0_uM
    return t_eval, capsid, free, inter, x


# --- control de conservacion de masa: si esto falla, el modelo esta roto ---
t_c, cap_c, free_c, int_c, _ = simulate_cascade(2.0, **q)
err = np.max(np.abs(cap_c + free_c + int_c - 1.0))
print(f"error maximo de conservacion de masa: {err:.2e}  "
      f"({'OK' if err < 1e-6 else 'PROBLEMA'})")

# --- perfil de energia libre por paso ---
for j in [2, 4, 5, 30, 83, 90]:
    c = contacts_for_step(j, q["n_c_nucleus"])
    dG = q["g_assoc_kcal"] + c * q["g_contact_kcal"]
    print(f"  paso -> {j:2d}-mero: {c} contacto(s), dG_add = {dG:+.1f} kcal/mol, "
          f"Kd = {np.exp(dG/RT):9.2e} uM")

## 4 · El barrido de concentración

Aquí aparece la predicción que distingue a este modelo de un simple diagrama: **más dímero no
implica mejor rendimiento**. Un modelo de ensamblaje al azar predice que el rendimiento crece de
forma monótona con la concentración; este modelo predice un colapso.

In [ ]:
# Barrido de concentración que produce la predicción falsable del modelo
D0_scan = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0])
yield_240 = []
inter_240 = []
for D0 in D0_scan:
    _, cap, free, inter, _ = simulate_cascade(D0, **q)
    yield_240.append(cap[-1])
    inter_240.append(inter[-1])
yield_240 = np.array(yield_240); inter_240 = np.array(inter_240)

i_opt = int(np.argmax(yield_240))
print("[D]0 (uM):", D0_scan)
print("capside:  ", np.round(yield_240, 2))
print("atrapado: ", np.round(inter_240, 2))
print(f"\noptimo en [D]0 = {D0_scan[i_opt]} uM  ->  rendimiento "
      f"{yield_240[i_opt]:.0%}")

## 5 · Las figuras

In [ ]:
fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.0, 4.1))

# ---- panel a: cinetica a tres concentraciones ----
for D0, c in [(1.0, "#17607a"), (2.0, "#0b6e5a"), (10.0, "#c0392b")]:
    t, cap, free, inter, _ = simulate_cascade(D0, **q)
    axA.plot(t, cap,   color=c, lw=2.0, label=f"[D]$_0$ = {D0:g} µM")
    axA.plot(t, inter, color=c, lw=1.2, ls="--", alpha=0.75)
axA.set_xlabel("tiempo (min)")
axA.set_ylabel("fracción de masa de subunidades")
axA.set_title("a   Cápside (sólido) vs intermedios atrapados (guion)",
              loc="left")
axA.set_ylim(-0.03, 1.03)
axA.legend(frameon=False, fontsize=8)

# ---- panel b: la curva con optimo ----
axB.plot(D0_scan, yield_240, "o-", color="#0b6e5a", lw=2.0, ms=6,
         label="cápside completa")
axB.plot(D0_scan, inter_240, "s--", color="#c0392b", lw=1.5, ms=5,
         alpha=0.8, label="atrapado en intermedios")
axB.axvline(D0_scan[i_opt], color="#d4a017", lw=1.4, ls=":")
axB.annotate(f"óptimo {D0_scan[i_opt]:g} µM\n({yield_240[i_opt]:.0%})",
             xy=(D0_scan[i_opt], yield_240[i_opt]),
             xytext=(D0_scan[i_opt] * 1.9, yield_240[i_opt] * 0.72),
             fontsize=8, color="#7a5c00",
             arrowprops=dict(arrowstyle="->", color="#d4a017", lw=1.0))
axB.set_xscale("log")
axB.set_xlabel(r"[D]$_0$ inicial de dímero (µM)")
axB.set_ylabel("fracción a los 240 min")
axB.set_title("b   La predicción falsable: existe un óptimo", loc="left")
axB.set_ylim(-0.03, 1.03)
axB.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig("M4_ensamblaje.png", dpi=300, bbox_inches="tight")
plt.show()
print("guardado: M4_ensamblaje.png")

## 6 · Composición de la trampa

Si el rendimiento colapsa, la masa debe encontrarse en algún lugar. La figura siguiente muestra
dónde y da la firma que debe buscarse por SEC-HPLC: un pico de intermedios de tamaño medio y no de
monómero libre.

In [ ]:
# Composición de la trampa cinética: distribución de tamaños.
# Se grafican solo los intermedios (1..89); la cápside completa (j=90) se
# reporta aparte porque, de incluirla, comprimiría el resto del eje.
fig, ax = plt.subplots(figsize=(6.6, 4.1))
sizes = np.arange(1, N_CAP + 1)

for D0, c in [(2.0, "#0b6e5a"), (10.0, "#c0392b")]:
    _, _, _, _, x = simulate_cascade(D0, **q)
    massfrac = sizes * x[:, -1] / D0        # fraccion de masa por tamano
    inter = massfrac[:-1]                   # intermedios 1..89
    ax.plot(sizes[:-1], inter, color=c, lw=1.9,
            label=f"[D]$_0$ = {D0:g} µM  ·  cápside {massfrac[-1]:.0%}")
    mean_size = np.sum(sizes[:-1] * inter) / max(np.sum(inter), 1e-12)
    print(f"[D]0 = {D0:5g} uM -> tamano medio de intermedio = {mean_size:5.1f} "
          f"dimeros | masa atrapada = {inter.sum():.2f} | "
          f"masa en capside = {massfrac[-1]:.2f}")

ax.set_xlabel("tamaño del intermedio (dímeros)")
ax.set_ylabel("fracción de masa a los 240 min")
ax.set_title("Dónde se acumula la masa cuando el ensamblaje se atasca",
             loc="left")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig("M4_distribucion.png", dpi=300, bbox_inches="tight")
plt.show()

## 7 · Sensibilidad del óptimo

Antes de proponer al laboratorio un barrido concreto conviene saber cuánto se desplaza el óptimo si
un parámetro de la literatura es incorrecto. Esto determina la rejilla de concentraciones que debe
cubrirse.

In [ ]:
# Sensibilidad: desplazamiento del óptimo cuando un parámetro es incorrecto
D0_fine = np.logspace(np.log10(0.3), np.log10(8.0), 26)

def optimum(**kw):
    pars = dict(q); pars.update(kw)
    ys = [simulate_cascade(d, **pars)[1][-1] for d in D0_fine]
    i = int(np.argmax(ys))
    return D0_fine[i], ys[i]

print(f"{'variante':<34}{'[D]0 optimo':>13}{'rendimiento':>13}")
base_D, base_y = optimum()
print(f"{'central (literatura)':<34}{base_D:>10.2f} µM{base_y:>12.0%}")
print(f"\n[nota] la rejilla GRUESA de la seccion 4 daba {D0_scan[i_opt]:g} µM "
      f"({yield_240[i_opt]:.0%}); esta rejilla FINA encuentra {base_D:.2f} µM "
      f"({base_y:.0%}).")
print("El óptimo es agudo: el rendimiento cae ~20 puntos si la concentración se desvía un")
print("factor 1.5, por lo que conviene muestrear con paso fino entre 1.5 y 3 uM.\n")
for label, kw in [
    ("g_contact = -3 (contacto debil)",  dict(g_contact_kcal=-3.0)),
    ("g_contact = -5 (contacto fuerte)", dict(g_contact_kcal=-5.0)),
    ("n_c = 3 (nucleo pequeno)",         dict(n_c_nucleus=3)),
    ("n_c = 6 (nucleo grande)",          dict(n_c_nucleus=6)),
    ("k_a = 0.4 (asociacion lenta)",     dict(k_a_uM_min=0.4)),
]:
    d, y = optimum(**kw)
    print(f"{label:<34}{d:>10.2f} µM{y:>12.0%}")

## Condiciones experimentales propuestas

Barrido que debe medirse por SEC-HPLC:

| | |
|---|---|
| **[D]₀** | 0.5, 1, 1.5, 2, 2.5, 3, 5, 10 µM |
| **tiempos** | 60, 120, 240 min |
| **réplicas** | 3 por condición |
| **condiciones** | 37 °C, fosfato 50 mM pH 7.2, NaCl 100 mM |
| **detener la reacción** | transferir a 4 °C de inmediato |

La rejilla debe cruzar el óptimo y llegar hasta la zona de colapso: si solo se mide hasta 2 µM no
es posible falsar el modelo. Como el óptimo es agudo (sección 7), se requieren puntos cada ~0.5 µM
entre 1.5 y 3 µM.

**Resultado esperado:** rendimiento máximo cerca de 2 µM y, por encima de 5 µM, masa en forma de
intermedios de tamaño medio y no de monómero libre. Si, en cambio, el rendimiento crece de forma
monótona hasta 10 µM, la hipótesis de nucleación con barrera es incorrecta y debe revisarse
`contacts_for_step`.

## Acoplamiento con M3

La entrada de M4 es la salida de M3, es decir, el pool de dímero. Con los valores centrales, M3
predice D_ss = 3.88 µM, por encima del óptimo de 2 µM. Reducir la temperatura de cultivo acerca el
dímero al óptimo (véase el notebook de M3), de modo que **la temperatura de inducción es una palanca
sobre el rendimiento de ensamblaje** y no solo un parámetro de crecimiento.